# 🗳️ Volební algoritmy

Jak přesně se z hlasů voličů stanou poslanecká křesla? V matematice a programování se tomuto problému říká **proporční zastoupení**. Na první pohled to zní jednoduše – spočítáme procenta a rozdělíme křesla. V praxi to ale naráží na problém: **nelze mít půlku poslance**.

V tomto notebooku si ukážeme algoritmy, které se v praxi používají pro přepočet hlasů na mandáty, a vyzkoušíme si je na reálných datech z voleb do Poslanecké sněmovny 2021.

## 1. Motivace a vstupní data

Budeme pracovat s daty z voleb do Poslanecké sněmovny v roce 2021. Celkem se rozděluje **200 mandátů** (křesel). Naším cílem bude naprogramovat algoritmus, který tyto mandáty spravedlivě rozdělí.

In [ ]:
# Oficiální celostátní výsledky z voleb do PS PČR 2021
volby_2021_hlasy = {
    "SPOLU": 1493905,
    "ANO": 1458140,
    "Piráti+STAN": 839776,
    "SPD": 513910,
    "Přísaha": 251562,
    "ČSSD": 250397,
    "KSČM": 193817,
    "Trikolora": 149720,
    "Zelení": 53583
}

CELKEM_MANDATU = 200

## 2. Volební klauzule (5% hranice)

Většina systémů používá tzv. **volební klauzuli**. V ČR musí strana získat alespoň 5 % ze všech odevzdaných hlasů, aby se vůbec dostala do sněmovny. Hlasy stran pod 5 % propadají.

Pojďme nejdřív filtrovat strany, které tuto klauzuli splnily.

In [ ]:
def aplikuj_klauzuli(hlasy_vsech, klauzule=0.05):
    celkovy_pocet_hlasu = sum(hlasy_vsech.values())
    minimalni_hlasy = celkovy_pocet_hlasu * klauzule
    
    postupujici_strany = {}
    for strana, pocet_hlasu in hlasy_vsech.items():
        if pocet_hlasu >= minimalni_hlasy:
            postupujici_strany[strana] = pocet_hlasu
            
    return postupujici_strany

# Strany nad 5 %
postupujici = aplikuj_klauzuli(volby_2021_hlasy)
print("Do rozdělování mandátů postupují:")
for strana, hlasy in postupujici.items():
    print(f"- {strana}: {hlasy:,} hlasů")

## 3. D'Hondtova metoda

D'Hondtova metoda je greedy algoritmus, který rozděluje mandáty **jeden po druhém**.

### Pravidla:
1. Každá strana má na začátku 0 mandátů.
2. V každém kole se pro každou stranu vypočítá tzv. **kvocient** (podíl):
   `Kvocient = počet_hlasů / (počet_získaných_mandátů + 1)`
3. Strana s největším kvocientem získá v tomto kole mandát.
4. Krok 2 a 3 se opakuje tak dlouho, dokud se nerozdělí všechny mandáty.

In [ ]:
def dhondt(hlasy, pocet_mandatu):
    # Krok 1: Inicializace - všechny strany mají 0 mandátů
    mandaty = {strana: 0 for strana in hlasy.keys()}
    
    # Smyčka přes všechny dostupné mandáty (rozdělujeme jeden po druhém)
    for i in range(pocet_mandatu):
        # Najdeme stranu s nejvyšším kvocientem
        nejvyssi_kvocient = -1
        vitezna_strana = None
        
        for strana, pocet_hlasu in hlasy.items():
            # Krok 2: Výpočet kvocientu pro danou stranu
            kvocient = pocet_hlasu / (mandaty[strana] + 1)
            
            if kvocient > nejvyssi_kvocient:
                nejvyssi_kvocient = kvocient
                vitezna_strana = strana
                
        # Krok 3: Přidělíme mandát vítězné straně
        mandaty[vitezna_strana] += 1
        
    return mandaty

In [ ]:
# Pojďme zkusit rozdělit 200 mandátů
rozdeleni_2021_dhondt = dhondt(postupujici, CELKEM_MANDATU)

print("Rozdělení 200 mandátů pomocí D'Hondtovy metody (klauzule 5%):")
for strana, m in sorted(rozdeleni_2021_dhondt.items(), key=lambda x: x[1], reverse=True):
    print(f"{strana}: {m} mandátů")

## 4. Vizualizace výsledků

Ke grafické reprezentaci můžeme využít knihovnu `matplotlib`. Zkusíme porovnat kolik % hlasů strana získala vs. kolik % mandátů obdržela. (D'Hondt je známý tím, že lehce **zvýhodňuje větší strany**).

In [ ]:
import matplotlib.pyplot as plt

# Procentuální zisk hlasů ze všech platných vs. mandáty (ze 200)
celkem_hlasu = sum(volby_2021_hlasy.values())
strany = list(rozdeleni_2021_dhondt.keys())

procenta_hlasu = [(volby_2021_hlasy[s] / celkem_hlasu) * 100 for s in strany]
procenta_mandatu = [(rozdeleni_2021_dhondt[s] / CELKEM_MANDATU) * 100 for s in strany]

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(strany))
width = 0.35

ax.bar([i - width/2 for i in x], procenta_hlasu, width, label='% Hlasů (z celku)')
ax.bar([i + width/2 for i in x], procenta_mandatu, width, label='% Mandátů')

ax.set_ylabel('Procenta (%)')
ax.set_title('Zisk hlasů vs zisk mandátů (D\'Hondt celostátně)')
ax.set_xticks(x)
ax.set_xticklabels(strany)
ax.legend()

plt.show()

## 5. Jak funguje český volební systém doopravdy?

Výpočet přes celostátní D'Hondt vyžadují prváci :) Proč? Protože český volební systém (změněný těsně před volbami v 2021) je mnohem složitější a rozděluje mandáty po krajích pomocí **dvou skrutinií (kol)**.

### 1. Skrutinium: Imperialiho kvóta v krajích
V každém ze 14 krajů se použije tzv. **kvóta**. Spočítá se kolik hlasů je 'cena za mandát' pomocí Imperialiho kvóty.
`Kvóta = Celkem platných hlasů v kraji / (Počet mandátů kraje + 2)`

Následně strana dostane tolik mandátů, kolikrát se kvóta vejde do jejích hlasů (zaokrouhleno dolů, neboli celočíselné dělení `//`). Toto ale nikdy nerozdělí všechny mandáty. Zbytek mandátů i takzvané 'zbytkové hlasy' jdou do druhého kola.

### 2. Skrutinium: Hagenbach-Bischoffova kvóta celostátně
Zbytky hlasů ze všech krajů se sečtou pro každou stranu. A nerozdělené mandáty se dají na „hromádku“. Následně se určí nová cena mandátu (Hagenbach-Bischoffova kvóta):
`Kvóta_2 = Součet všech zbytkových hlasů / (Počet nerozdělených mandátů + 1)`
Strany opět dostanou křesla za celé násobky, a případné zbylé se rozdají těm, kdo mají největší nerozdělený zbytek hlasů.

In [ ]:
# Ukázka 1. Skrutinia (zjednodušeně pro jeden 'kraj')
hlasy_v_kraji = {"SPOLU": 250000, "ANO": 220000, "Piráti+STAN": 100000, "SPD": 60000}
kraj_mandaty = 20

# 1. Imperialiho kvóta
celkem_platnych = sum(hlasy_v_kraji.values())
kvota = celkem_platnych // (kraj_mandaty + 2) # Celočíselné dělení

print(f"Celkem hlasů: {celkem_platnych}, Cena za 1 mandát (kvóta): {kvota}\n")

rozdeleno_mandatu = 0
zbytky_hlasu = {}

for strana, hlasy in hlasy_v_kraji.items():
    ziskano_mandatu = hlasy // kvota # Kolikrát se kvóta vejde do hlasů
    zbytek = hlasy % kvota           # Co straně zbylo?
    
    rozdeleno_mandatu += ziskano_mandatu
    zbytky_hlasu[strana] = zbytek
    
    print(f"{strana}: {ziskano_mandatu} mandátů (Zbytek hlasů: {zbytek})")

print(f"\nRozděleno celkem mandátů v 1. kole: {rozdeleno_mandatu} z {kraj_mandaty}.")
print(f"Zbývají {kraj_mandaty - rozdeleno_mandatu} mandáty do druhého (celostátního) kola!")